# 04 - Sémantique : du graphe RDF aux requêtes SPARQL

On décrit le domaine avec l'ontologie `ontology/dvf.ttl` (classes, relations, propriétés), puis on transforme les ventes de la silver **d'un seul département** en triplets RDF, et on les interroge en SPARQL.

Un seul département : le graphe reste petit (quelques centaines de milliers de triplets au plus), ce qui suffit à montrer le principe. Les requêtes sont en mémoire avec `rdflib`, sans serveur à lancer.

In [1]:
import subprocess
import sys

try:
    import rdflib
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "rdflib"])
    import rdflib

sys.path.append("/home/jovyan/work")

import os
from decimal import Decimal

import pandas as pd
from pyspark.sql import functions as F
from rdflib import Graph, Literal, Namespace, RDF

from outils.source import creer_session, LAKE

ANNEE = 2025
SILVER = f"{LAKE}/silver/ventes_dvf_{ANNEE}"
ONTOLOGIE = "/home/jovyan/work/ontology/dvf.ttl"
DEPARTEMENT = "33"        # un seul département pour limiter la taille du graphe

DVF = Namespace("https://example.org/dvf#")      # le vocabulaire (l'ontologie)
ID = Namespace("https://example.org/dvf/id/")    # les individus (les ventes, communes...)

## 1. Charger l'ontologie

In [2]:
g = Graph()
g.parse(ONTOLOGIE, format="turtle")
g.bind("dvf", DVF)
print(len(g), "triplets dans l'ontologie")

for classe in sorted(str(s).split("#")[1] for s in g.subjects(RDF.type, rdflib.OWL.Class)):
    print(" classe :", classe)

80 triplets dans l'ontologie
 classe : Appartement
 classe : Commune
 classe : Departement
 classe : Logement
 classe : Maison
 classe : Mutation
 classe : Parcelle


## 2. Lire les ventes d'un département dans la silver

In [3]:
spark = creer_session("semantique")
ventes = (spark.read.format("delta").load(SILVER)
          .filter(F.col("code_departement") == DEPARTEMENT)
          .toPandas())
spark.stop()
print(len(ventes), "ventes dans le département", DEPARTEMENT)
ventes.head()

Spark 4.2.0 (Hadoop 3.5.0) + Delta Lake + data lake S3. Le premier démarrage télécharge des jars : plusieurs minutes.
Session prête.
22128 ventes dans le département 33


,id_mutation,date_mutation,annee,mois,valeur_fonciere,type_local,surface_bati,nb_pieces,code_departement,code_commune,nom_commune,id_parcelle,longitude,latitude,prix_m2
0,2025-401881,2025-01-13,2025,1,285000.0,Maison,109.0,6,33,33318,Pessac,33318000AY0653,-0.665115,44.807667,2614.678899
1,2025-402013,2025-02-11,2025,2,283000.0,Maison,64.0,3,33,33063,Bordeaux,33063000CR0031,-0.567460,44.823018,4421.875000
2,2025-402337,2025-02-07,2025,2,317750.0,Maison,84.0,4,33,33449,Saint Medard en Jalles,33449000DX0278,-0.761980,44.892476,3782.738095
3,2025-402356,2025-02-14,2025,2,175000.0,Appartement,57.0,3,33,33200,Le Haillan,33200000AS0379,-0.674111,44.868178,3070.175439
4,2025-402405,2025-02-12,2025,2,244100.0,Appartement,80.0,5,33,33063,Bordeaux,33063000NM0046,-0.620222,44.839716,3051.250000


## 3. Transformer chaque vente en triplets
Une ligne de la silver devient un petit sous-graphe : une **mutation** qui porte sur un **logement** (maison ou appartement), situé sur une **parcelle**, dans une **commune**, qui appartient à un **département**. Les types viennent de l'ontologie.

In [4]:
def decimal(x, chiffres=2):
    return Literal(Decimal(str(round(float(x), chiffres))))

for r in ventes.itertuples(index=False):
    m = ID[f"mutation/{r.id_mutation}"]
    l = ID[f"logement/{r.id_mutation}"]
    p = ID[f"parcelle/{r.id_parcelle}"]
    c = ID[f"commune/{r.code_commune}"]
    d = ID[f"departement/{r.code_departement}"]

    g.add((m, RDF.type, DVF.Mutation))
    g.add((m, DVF.portesSur, l))
    g.add((m, DVF.valeurFonciere, decimal(r.valeur_fonciere)))
    g.add((m, DVF.prixAuM2, decimal(r.prix_m2)))
    if pd.notna(r.date_mutation):
        g.add((m, DVF.dateMutation, Literal(r.date_mutation)))

    g.add((l, RDF.type, DVF.Maison if r.type_local == "Maison" else DVF.Appartement))
    g.add((l, DVF.situeSurParcelle, p))
    g.add((l, DVF.surfaceBati, decimal(r.surface_bati)))
    if pd.notna(r.nb_pieces):
        g.add((l, DVF.nombrePieces, Literal(int(r.nb_pieces))))

    g.add((p, RDF.type, DVF.Parcelle))
    g.add((p, DVF.situeeDans, c))
    if pd.notna(r.longitude) and pd.notna(r.latitude):
        g.add((p, DVF.longitude, decimal(r.longitude, 6)))
        g.add((p, DVF.latitude, decimal(r.latitude, 6)))

    g.add((c, RDF.type, DVF.Commune))
    g.add((c, DVF.codeCommune, Literal(r.code_commune)))
    g.add((c, DVF.nomCommune, Literal(r.nom_commune)))
    g.add((c, DVF.appartientA, d))

    g.add((d, RDF.type, DVF.Departement))
    g.add((d, DVF.codeDepartement, Literal(r.code_departement)))

print(len(g), "triplets au total")

os.makedirs("/home/jovyan/work/data/rdf", exist_ok=True)
g.serialize(f"/home/jovyan/work/data/rdf/ventes_{DEPARTEMENT}.ttl", format="turtle")

269418 triplets au total


<Graph identifier=N3ad10c3f72d44cf692bbc1bf0aab85db (<class 'rdflib.graph.Graph'>)>

## 4. Requêtes SPARQL
Une petite fonction exécute une requête et renvoie un tableau.

In [20]:
PREFIXES = """
PREFIX dvf:  <https://example.org/dvf#>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
"""

def valeur(v):
    x = v.toPython() if v is not None else None
    return float(x) if isinstance(x, Decimal) else x

def sparql(requete):
    resultat = g.query(PREFIXES + requete)
    lignes = [[valeur(v) for v in ligne] for ligne in resultat]
    return pd.DataFrame(lignes, columns=[str(v) for v in resultat.vars]).round(1)

### Requête 1 : où se concentre l'activité ?
*Pour Claire : dans quelles communes du département se font le plus de ventes ?* On suit la chaîne mutation → logement → parcelle → commune.

In [14]:
sparql("""
SELECT ?commune (COUNT(?m) AS ?nb_ventes)
WHERE {
  ?m a dvf:Mutation ;
     dvf:portesSur ?l .
  ?l dvf:situeSurParcelle ?p .
  ?p dvf:situeeDans ?c .
  ?c dvf:nomCommune ?commune .
}
GROUP BY ?c ?commune
ORDER BY DESC(?nb_ventes)
LIMIT 10
""")

,commune,nb_ventes
0,Bordeaux,4505
1,Merignac,1114
2,Pessac,697
3,Talence,628
4,Villenave D Ornon,513
5,La Teste-de-Buch,499
6,Arcachon,461
7,Le Bouscat,446
8,Begles,425
9,Libourne,382


### Requête 2 : quelles communes actives sont moins chères que le département ?
*Pour Claire : où ouvrir une agence dans le département retenu ?* On garde les communes qui ont assez de ventes et dont le prix au m² moyen est **inférieur** à celui du département. SPARQL n'a pas de médiane : on compare des **moyennes**, ce qui diffère de la gold.

In [15]:
MIN_VENTES_COMMUNE = 20    # en dessous, la moyenne d'une commune n'est pas fiable : à justifier

sparql(f"""
SELECT ?commune ?nb_ventes ?prix_m2_moyen ?moyenne_departement
WHERE {{
  {{
    SELECT ?c ?commune (COUNT(?m) AS ?nb_ventes) (AVG(?prix) AS ?prix_m2_moyen)
    WHERE {{
      ?m dvf:prixAuM2 ?prix ;
         dvf:portesSur ?l .
      ?l dvf:situeSurParcelle ?p .
      ?p dvf:situeeDans ?c .
      ?c dvf:nomCommune ?commune .
    }}
    GROUP BY ?c ?commune
    HAVING (COUNT(?m) >= {MIN_VENTES_COMMUNE})
  }}
  {{
    SELECT (AVG(?prix) AS ?moyenne_departement)
    WHERE {{ ?m dvf:prixAuM2 ?prix . }}
  }}
  FILTER (?prix_m2_moyen < ?moyenne_departement)
}}
ORDER BY DESC(?nb_ventes)
LIMIT 10
""")

,commune,nb_ventes,prix_m2_moyen,moyenne_departement
0,Merignac,1114,3612.4,3642.2
1,Villenave D Ornon,513,3332.3,3642.2
2,Begles,425,3640.4,3642.2
3,Libourne,382,2397.7,3642.2
4,Cenon,374,3061.0,3642.2
5,Saint Medard en Jalles,350,3512.0,3642.2
6,Eysines,330,3426.5,3642.2
7,Gradignan,320,3571.7,3642.2
8,Lormont,231,2617.7,3642.2
9,Le Haillan,188,3604.8,3642.2


### Requête 3 : comment l'activité évolue-t-elle dans l'année ?
*Pour Claire : quelle est la saisonnalité des ventes dans ce département ?* Le mois est extrait de la date par la fonction `MONTH`.

In [9]:
sparql("""
SELECT ?mois (COUNT(?m) AS ?nb_ventes)
WHERE {
  ?m a dvf:Mutation ;
     dvf:dateMutation ?date .
  BIND (MONTH(?date) AS ?mois)
}
GROUP BY ?mois
ORDER BY ?mois
""")

,mois,nb_ventes
0,1,1491
1,2,1564
2,3,1636
3,4,2237
4,5,1262
5,6,1954
6,7,2447
7,8,1655
8,9,2037
9,10,2054


### Requête 4 : maisons ou appartements ? (l'ontologie à l'œuvre)
Le type de logement n'est pas écrit dans la requête : on le **découvre** grâce à la hiérarchie de l'ontologie (`Maison` et `Appartement` sont des sous-classes de `Logement`). Si l'on ajoutait une nouvelle sous-classe dans l'ontologie, la requête la prendrait en compte sans modification.

In [12]:
sparql("""
SELECT ?type (COUNT(?m) AS ?nb_ventes) (AVG(?prix) AS ?prix_m2_moyen)
WHERE {
  ?type rdfs:subClassOf dvf:Logement .
  ?l a ?type .
  ?m dvf:portesSur ?l ;
     dvf:prixAuM2 ?prix .
}
GROUP BY ?type
""")

,type,nb_ventes,prix_m2_moyen
0,https://example.org/dvf#Maison,13095,3490.3
1,https://example.org/dvf#Appartement,9033,3862.5
